# Automated Audio Restoration Pipeline
This notebook processes audio using deep learning models to remove room echo, reverb, and degradation. It executes in isolated subprocesses and automatically crossfades chunks to prevent VRAM crashes and seam artifacts.

**Instructions:**
* Ensure your runtime is set to **T4 GPU** or higher (`Runtime > Change runtime type`).
* Run **Cell 1** to mount Google Drive and configure your settings.
* Run **Cell 2** to execute the pipeline.

In [ ]:
import os
import sys
import subprocess
from google.colab import drive

print("Requesting Google Drive access...")
drive.mount('/content/drive')

print("Installing dependencies... This may take a moment.")
install_reqs = subprocess.run(
    [sys.executable, "-m", "pip", "install", "voicefixer", "audio-separator[gpu]", "ipywidgets", "tqdm", "psutil"],
    capture_output=True,
    text=True
)
if install_reqs.returncode != 0:
    print("Error installing dependencies:")
    print(install_reqs.stderr)
else:
    from IPython.display import clear_output
    clear_output()
    print("Drive mounted and dependencies installed successfully.")

import ipywidgets as widgets
from IPython.display import display
import glob

# Global Configuration Storage
config = {
    "file_path": "",
    "model": "",
    "option": ""
}

# UI Components
dir_input = widgets.Text(value='/content/drive/MyDrive', description='Drive Dir:', layout=widgets.Layout(width='50%'))
refresh_btn = widgets.Button(description='Scan Directory', button_style='info')
file_dropdown = widgets.Dropdown(description='Audio File:', layout=widgets.Layout(width='50%'))

model_dropdown = widgets.Dropdown(
    options=['UVR5 (De-Echo)', 'VoiceFixer (General Restore)'],
    description='Model:',
    layout=widgets.Layout(width='50%')
)

option_dropdown = widgets.Dropdown(description='Option:', layout=widgets.Layout(width='50%'))
ready_btn = widgets.Button(description='Lock Configuration', button_style='success')
output_log = widgets.Output()

def update_files(b):
    search_dir = dir_input.value
    if not os.path.exists(search_dir):
        with output_log:
            print(f"Directory {search_dir} not found.")
        return
    
    # Find common audio types
    types = ('*.wav', '*.mp3', '*.m4a', '*.flac')
    files = []
    for ext in types:
        files.extend(glob.glob(os.path.join(search_dir, ext)))
    
    files.sort(key=os.path.getmtime, reverse=True)
    if not files:
        file_dropdown.options = ['No audio files found']
    else:
        file_dropdown.options = [(os.path.basename(f), f) for f in files]

def update_options(change):
    if change['new'] == 'UVR5 (De-Echo)':
        option_dropdown.options = [('Aggressive', 'UVR-De-Echo-Aggressive.pth'), ('Normal', 'UVR-De-Echo-Normal.pth')]
    else:
        option_dropdown.options = [('Mode 0 (Standard)', '0'), ('Mode 1 (High-Freq Filter)', '1')]

def lock_config(b):
    with output_log:
        clear_output()
        config['file_path'] = file_dropdown.value
        config['model'] = model_dropdown.value
        config['option'] = option_dropdown.value
        if config['file_path'] == 'No audio files found' or not config['file_path']:
            print("Error: No valid file selected.")
            return
        print("Configuration Locked:")
        print(f"File: {config['file_path']}")
        print(f"Model: {config['model']}")
        print(f"Option: {config['option']}")
        print("\nYou may now run Cell 2.")

refresh_btn.on_click(update_files)
model_dropdown.observe(update_options, names='value')
ready_btn.on_click(lock_config)

# Initialize states
update_options({'new': model_dropdown.value})
update_files(None)

display(dir_input, refresh_btn, file_dropdown, model_dropdown, option_dropdown, ready_btn, output_log)


In [ ]:
import json
import math
import shutil
from tqdm.notebook import tqdm
from IPython.display import Audio, display

if not config.get('file_path'):
    raise ValueError("Configuration not locked. Please run Cell 1 and click 'Lock Configuration'.")

input_file = config['file_path']
work_dir = "/content/processing"
os.makedirs(work_dir, exist_ok=True)

print("\n--- 1. Probing Original Metadata ---")
probe_cmd = ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", input_file]
probe_res = subprocess.run(probe_cmd, capture_output=True, text=True)
if probe_res.returncode != 0:
    print("Error probing file:", probe_res.stderr)
    raise RuntimeError("ffprobe failed.")

probe_data = json.loads(probe_res.stdout)
audio_stream = next(s for s in probe_data['streams'] if s['codec_type'] == 'audio')
sample_rate = audio_stream.get('sample_rate', '44100')
print(f"Detected Sample Rate: {sample_rate} Hz")

print("\n--- 2. Format Normalization ---")
base_wav = os.path.join(work_dir, "normalized.wav")
norm_cmd = ["ffmpeg", "-y", "-i", input_file, "-acodec", "pcm_s16le", "-ar", sample_rate, base_wav]
norm_res = subprocess.run(norm_cmd, capture_output=True, text=True)
if norm_res.returncode != 0:
    print(norm_res.stderr)
    raise RuntimeError("FFmpeg normalization failed.")

print("\n--- 3. Overlap Chunking ---")
duration = float(audio_stream.get('duration', 0))
if duration == 0:
    probe_dur = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", base_wav], capture_output=True, text=True)
    duration = float(probe_dur.stdout.strip())

chunk_len = 180  # 3 minutes
overlap = 10     # 10 second overlap
step = chunk_len - overlap
num_chunks = math.ceil(duration / step)

raw_chunks = []
for i in range(num_chunks):
    start_time = i * step
    chunk_file = os.path.join(work_dir, f"raw_chunk_{i:03d}.wav")
    chunk_cmd = ["ffmpeg", "-y", "-ss", str(start_time), "-t", str(chunk_len), "-i", base_wav, chunk_file]
    subprocess.run(chunk_cmd, check=True, capture_output=True)
    raw_chunks.append(chunk_file)
print(f"Created {len(raw_chunks)} chunks with {overlap}s overlaps.")
os.remove(base_wav)

print("\n--- 4. Processing Loop (Isolated Subprocesses) ---")
processed_chunks = []

for i, chunk in enumerate(tqdm(raw_chunks, desc="Processing Audio Chunks")):
    out_file = os.path.join(work_dir, f"fixed_chunk_{i:03d}.wav")
    
    if "VoiceFixer" in config['model']:
        cmd = ["voicefixer", "--infile", chunk, "--outfile", out_file, "--mode", config['option']]
    else:
        # UVR5 logic via audio-separator
        cmd = ["audio-separator", chunk, "--model_filename", config['option'], "--output_dir", work_dir, "--output_format", "wav"]
    
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f"Model inference failed on chunk {i}")
    
    if "UVR5" in config['model']:
        # Locate generated instrumental/reverb-free file
        gen_files = glob.glob(os.path.join(work_dir, "*(Instrumental)*.wav"))
        if gen_files:
            os.rename(gen_files[0], out_file)
            # Cleanup leftover vocal files
            for leftover in glob.glob(os.path.join(work_dir, "*(Vocals)*.wav")):
                os.remove(leftover)
    
    processed_chunks.append(out_file)
    os.remove(chunk)  # Immediate garbage collection

print("\n--- 5. Iterative Crossfade Concatenation ---")
concat_wav = os.path.join(work_dir, "final_concat.wav")

if len(processed_chunks) == 1:
    os.rename(processed_chunks[0], concat_wav)
else:
    current_mix = processed_chunks[0]
    for idx in range(1, len(processed_chunks)):
        next_mix = os.path.join(work_dir, f"temp_mix_{idx}.wav")
        filter_complex = f"[0:a]aresample={sample_rate}[a0];[1:a]aresample={sample_rate}[a1];[a0][a1]acrossfade=d={overlap}"
        
        concat_cmd = [
            "ffmpeg", "-y", 
            "-i", current_mix, 
            "-i", processed_chunks[idx], 
            "-filter_complex", filter_complex, 
            "-ar", str(sample_rate), 
            next_mix
        ]
        concat_res = subprocess.run(concat_cmd, capture_output=True, text=True)
        if concat_res.returncode != 0:
            print(concat_res.stderr)
            raise RuntimeError(f"Crossfade concatenation failed at chunk {idx}.")
        
        current_mix = next_mix
        
    os.rename(current_mix, concat_wav)

print("\n--- 6. Final Encoding & Metadata Injection ---")
orig_name, orig_ext = os.path.splitext(os.path.basename(input_file))
model_slug = config['model'].split()[0]
opt_slug = config['option'].split('.')[0].replace(' ', '_')
final_filename = f"{orig_name}_{model_slug}_{opt_slug}{orig_ext}"
final_output_path = os.path.join(os.path.dirname(input_file), final_filename)

meta_cmd = [
    "ffmpeg", "-y", 
    "-i", concat_wav,
    "-i", input_file,
    "-map", "0:a", 
    "-map_metadata", "1", 
    "-c:a", audio_stream.get('codec_name', 'aac'),
    final_output_path
]
meta_res = subprocess.run(meta_cmd, capture_output=True, text=True)
if meta_res.returncode != 0:
    print("Warning: Codec copy failed, attempting fallback to uncompressed WAV.")
    shutil.copy(concat_wav, final_output_path)

print("\n--- 7. Delivery & Cleanup ---")
shutil.rmtree(work_dir)
print(f"Successfully saved to: {final_output_path}")

display(Audio(final_output_path))
